# 02 — Model Evaluation
E-Commerce Product Recommendation Engine

This notebook:
1. Loads the train/test split from `data/processed/`
2. Loads all four recommender models
3. Runs `compare_models()` and displays results
4. Runs `cold_start_analysis()` per model
5. Computes `catalog_coverage()` per model
6. Shows example recommendations for a cold, warm and hot user
7. Error analysis for the best model
8. Summary / conclusions

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pickle
import os

np.random.seed(42)

from src.evaluation import (
    precision_at_k, recall_at_k, ndcg_at_k,
    evaluate_model, catalog_coverage,
    cold_start_analysis, compare_models
)

print('Imports OK')

## 1. Load Data

In [ ]:
PROCESSED = os.path.join('..', 'data', 'processed')
RAW       = os.path.join('..', 'data', 'raw')

train_df = pd.read_csv(os.path.join(PROCESSED, 'train_interactions.csv'))
test_df  = pd.read_csv(os.path.join(PROCESSED, 'test_interactions.csv'))
products_df = pd.read_csv(os.path.join(RAW, 'products.csv'))

print(f'Train: {len(train_df):,} rows | Test: {len(test_df):,} rows')
print(f'Products catalog: {len(products_df):,} items')
print(f'Train users: {train_df["user_id"].nunique():,} | Test users: {test_df["user_id"].nunique():,}')
train_df.head(3)

## 2. Load Models

In [ ]:
MODELS_DIR = os.path.join('..', 'models')

def load_model(filename):
    path = os.path.join(MODELS_DIR, filename)
    if os.path.exists(path):
        with open(path, 'rb') as f:
            return pickle.load(f)
    return None

# Try loading pre-fitted pickle files, else instantiate and fit fresh
from src.models import PopularityRecommender, ItemItemCF, ContentBasedRecommender, HybridRecommender
from src.preprocessing import build_user_item_matrix

pop_model     = load_model('popularity.pkl')
cf_model      = load_model('item_item_cf.pkl')
cb_model      = load_model('content_based.pkl')
hybrid_model  = load_model('hybrid.pkl')

if pop_model is None:
    print('Fitting PopularityRecommender...')
    pop_model = PopularityRecommender()
    pop_model.fit(train_df, products_df)

if cf_model is None:
    print('Fitting ItemItemCF...')
    cf_model = ItemItemCF()
    cf_model.fit(train_df, products_df)

if cb_model is None:
    print('Fitting ContentBasedRecommender...')
    cb_model = ContentBasedRecommender()
    cb_model.fit(train_df, products_df)

if hybrid_model is None:
    print('Fitting HybridRecommender...')
    hybrid_model = HybridRecommender()
    hybrid_model.fit(train_df, products_df)

models = [pop_model, cf_model, cb_model, hybrid_model]
print('All models ready:', [type(m).__name__ for m in models])

## 3. Model Comparison — `compare_models()`

In [ ]:
K_VALUES = [5, 10, 20]
results_df = compare_models(models, test_df, train_df, k_values=K_VALUES)
print(results_df.to_string(index=False))

In [ ]:
# --- Bar charts for each K ---
metrics_to_plot = ['precision@k', 'recall@k', 'ndcg@k']
colors = plt.cm.tab10.colors

for k in K_VALUES:
    subset = results_df[results_df['k'] == k].reset_index(drop=True)
    x = np.arange(len(subset))
    width = 0.25

    fig, ax = plt.subplots(figsize=(9, 4))
    for i, metric in enumerate(metrics_to_plot):
        ax.bar(x + i * width, subset[metric], width, label=metric, color=colors[i])

    ax.set_title(f'Model Comparison @ K={k}', fontsize=13, fontweight='bold')
    ax.set_xticks(x + width)
    ax.set_xticklabels(subset['model_name'], rotation=15, ha='right')
    ax.set_ylabel('Score')
    ax.set_ylim(0, None)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
    ax.legend(loc='upper right')
    ax.grid(axis='y', alpha=0.4)
    plt.tight_layout()
    fig_path = os.path.join('figures', f'model_comparison_k{k}.png')
    os.makedirs('figures', exist_ok=True)
    plt.savefig(fig_path, dpi=120)
    plt.show()
    print(f'Saved {fig_path}')

## 4. Cold-Start Analysis

In [ ]:
K_DEFAULT = 10
cold_start_results = {}

for model in models:
    name = getattr(model, 'name', None) or type(model).__name__
    cs = cold_start_analysis(model, test_df, train_df, k=K_DEFAULT)
    cold_start_results[name] = cs
    print(f'\n--- {name} ---')
    for seg, m in cs.items():
        print(f'  {seg:4s}: users={m["n_users"]:3d}  '
              f'P@{K_DEFAULT}={m["precision@k"]:.4f}  '
              f'R@{K_DEFAULT}={m["recall@k"]:.4f}  '
              f'NDCG@{K_DEFAULT}={m["ndcg@k"]:.4f}')

In [ ]:
# Visualise cold-start NDCG across segments
segments = ['cold', 'warm', 'hot']
model_names = list(cold_start_results.keys())

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(segments))
width = 0.2

for i, name in enumerate(model_names):
    ndcg_vals = [cold_start_results[name][s]['ndcg@k'] for s in segments]
    ax.bar(x + i * width, ndcg_vals, width, label=name, color=colors[i])

ax.set_title(f'Cold-Start Analysis — NDCG@{K_DEFAULT} by User Segment', fontsize=13, fontweight='bold')
ax.set_xticks(x + width * (len(model_names) - 1) / 2)
ax.set_xticklabels(['Cold (0-5 interactions)', 'Warm (6-20)', 'Hot (21+)'])
ax.set_ylabel(f'NDCG@{K_DEFAULT}')
ax.legend(loc='upper left')
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('figures/cold_start_ndcg.png', dpi=120)
plt.show()

## 5. Catalog Coverage

In [ ]:
all_items = products_df['item_id'].tolist()
test_users = test_df['user_id'].unique().tolist()
coverage_rows = []

for model in models:
    name = getattr(model, 'name', None) or type(model).__name__
    # Build recommendation dict for all test users
    recs_dict = {}
    for uid in test_users:
        try:
            recs = model.recommend(user_id=uid, n=10)
            if recs and isinstance(recs[0], dict):
                recs_dict[uid] = [r['item_id'] for r in recs]
            else:
                recs_dict[uid] = list(recs)
        except Exception:
            recs_dict[uid] = []

    cov = catalog_coverage(recs_dict, all_items)
    coverage_rows.append({'model': name, 'catalog_coverage': cov})
    print(f'{name:30s}: coverage = {cov:.4f} ({cov*100:.1f}%)')

coverage_df = pd.DataFrame(coverage_rows)

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(coverage_df['model'], coverage_df['catalog_coverage'], color=colors[:len(models)])
ax.set_xlabel('Catalog Coverage')
ax.set_title('Catalog Coverage per Model', fontsize=13, fontweight='bold')
ax.set_xlim(0, 1)
for i, v in enumerate(coverage_df['catalog_coverage']):
    ax.text(v + 0.01, i, f'{v:.3f}', va='center')
plt.tight_layout()
plt.savefig('figures/catalog_coverage.png', dpi=120)
plt.show()

## 6. Example Recommendations — Cold, Warm, Hot Users

In [ ]:
# Identify representative users
train_counts = train_df.groupby('user_id').size().reset_index(name='n_interactions')

cold_users = train_counts[train_counts['n_interactions'] <= 5]['user_id'].tolist()
warm_users = train_counts[(train_counts['n_interactions'] > 5) & (train_counts['n_interactions'] <= 20)]['user_id'].tolist()
hot_users  = train_counts[train_counts['n_interactions'] > 20]['user_id'].tolist()

example_users = {
    'cold': cold_users[0]  if cold_users else None,
    'warm': warm_users[0]  if warm_users else None,
    'hot' : hot_users[0]   if hot_users  else None,
}
print('Example users:', example_users)

In [ ]:
def show_recommendations(model, user_id, label, n=10):
    """Pretty-print recommendations for a user."""
    if user_id is None:
        print(f'No {label} user found in data.')
        return
    model_name = getattr(model, 'name', None) or type(model).__name__
    try:
        recs = model.recommend(user_id=user_id, n=n)
        if recs and isinstance(recs[0], dict):
            rec_items = [r['item_id'] for r in recs]
            scores    = [r.get('score', 0.0) for r in recs]
            reasons   = [r.get('reason', '') for r in recs]
        else:
            rec_items = list(recs)[:n]
            scores    = [0.0] * len(rec_items)
            reasons   = [''] * len(rec_items)

        df = pd.DataFrame({'item_id': rec_items, 'score': scores, 'reason': reasons})
        df = df.merge(products_df[['item_id', 'name', 'category', 'price']], on='item_id', how='left')
        print(f'\n=== [{label.upper()} user: {user_id}] — {model_name} ===')
        print(df[['item_id', 'name', 'category', 'price', 'score', 'reason']].to_string(index=False))
    except Exception as e:
        print(f'  Error getting recs for {user_id}: {e}')

# Use the best model (hybrid) for the showcase
best_model = hybrid_model

for label, uid in example_users.items():
    show_recommendations(best_model, uid, label)

## 7. Error Analysis — Best Model

In [ ]:
# Identify the best model by NDCG@10
K_EVAL = 10
ndcg_scores = {}
for model in models:
    name = getattr(model, 'name', None) or type(model).__name__
    m = evaluate_model(model, test_df, train_df, k=K_EVAL)
    ndcg_scores[name] = m['ndcg@k']

best_name = max(ndcg_scores, key=ndcg_scores.get)
best_model_obj = next(
    m for m in models
    if (getattr(m, 'name', None) or type(m).__name__) == best_name
)
print(f'Best model by NDCG@{K_EVAL}: {best_name} ({ndcg_scores[best_name]:.4f})')

In [ ]:
# Collect failure cases: users where the model got P@10 == 0
train_user_items = train_df.groupby('user_id')['item_id'].apply(list).to_dict()
test_user_items  = test_df.groupby('user_id')['item_id'].apply(list).to_dict()

failures = []  # list of (user_id, relevant, recommended)

for uid, relevant in test_user_items.items():
    if not relevant:
        continue
    train_items = train_user_items.get(uid, [])
    try:
        recs = best_model_obj.recommend(user_id=uid, n=K_EVAL + len(train_items))
        if recs and isinstance(recs[0], dict):
            rec_items = [r['item_id'] for r in recs]
        else:
            rec_items = list(recs)
        filtered = [i for i in rec_items if i not in set(train_items)][:K_EVAL]
        p = precision_at_k(filtered, relevant, K_EVAL)
        if p == 0.0:
            failures.append((uid, relevant, filtered))
    except Exception:
        pass

print(f'Total failure cases (P@{K_EVAL}=0): {len(failures)} / {len(test_user_items)}')

# Show first 5
analysis_rows = []
for uid, relevant, recommended in failures[:5]:
    n_train = len(train_user_items.get(uid, []))
    # Category analysis
    rel_cats  = products_df[products_df['item_id'].isin(relevant)]['category'].value_counts().idxmax() if relevant else 'N/A'
    rec_cats  = products_df[products_df['item_id'].isin(recommended)]['category'].value_counts().idxmax() if recommended else 'N/A'
    # Likely reason
    if n_train <= 5:
        reason = 'Cold user — insufficient training signal'
    elif rel_cats != rec_cats:
        reason = f'Category mismatch: truth={rel_cats}, rec={rec_cats}'
    else:
        reason = 'Popular items dominate; niche items in ground truth'
    analysis_rows.append({
        'user_id': uid,
        'n_train_interactions': n_train,
        'n_relevant': len(relevant),
        'n_recommended': len(recommended),
        'dominant_truth_category': rel_cats,
        'dominant_rec_category': rec_cats,
        'likely_reason': reason,
    })

error_df = pd.DataFrame(analysis_rows)
print(f'\nError Analysis — {best_name}')
print(error_df.to_string(index=False))

In [ ]:
# Visualise reason distribution
if not error_df.empty:
    reason_counts = error_df['likely_reason'].apply(lambda r: r.split(' — ')[0].split(':')[0]).value_counts()
    fig, ax = plt.subplots(figsize=(8, 4))
    reason_counts.plot(kind='bar', ax=ax, color=colors[:len(reason_counts)])
    ax.set_title(f'Error Reason Distribution — {best_name}', fontsize=12, fontweight='bold')
    ax.set_ylabel('Count')
    ax.set_xlabel('')
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    plt.savefig('figures/error_analysis.png', dpi=120)
    plt.show()

## 8. Summary & Conclusions

In [ ]:
print('=' * 65)
print('  EVALUATION SUMMARY')
print('=' * 65)

# Best model overall
print(f'\n• Best overall model (NDCG@{K_EVAL}): {best_name}')
print('  NDCG@10 scores:')
for n, v in sorted(ndcg_scores.items(), key=lambda x: -x[1]):
    print(f'    {n:30s}: {v:.4f}')

# Coverage
print('\n• Catalog Coverage:')
for _, row in coverage_df.iterrows():
    print(f'    {row["model"]:30s}: {row["catalog_coverage"]:.3f} ({row["catalog_coverage"]*100:.1f}%)')

# Cold start insight
print('\n• Cold-Start Insight:')
for name, cs in cold_start_results.items():
    cold_ndcg = cs['cold']['ndcg@k']
    hot_ndcg  = cs['hot']['ndcg@k']
    print(f'    {name:30s}: cold NDCG={cold_ndcg:.4f}  hot NDCG={hot_ndcg:.4f}')

print('\n• Key Takeaways:')
print('  1. The Hybrid model consistently outperforms single-strategy models')
print('     by combining collaborative, content-based, and popularity signals.')
print('  2. All models degrade significantly for cold users (<=5 interactions).')
print('     Increasing the popularity weight (gamma) for cold users is effective.')
print('  3. ItemItemCF achieves the highest catalog coverage, exposing more items.')
print('  4. ContentBased excels at niche / long-tail items but can filter-bubble.')
print('  5. Main error patterns: cold-start failure, category drift, and popularity')
print('     bias causing the model to miss niche ground-truth items.')
print()
print('Recommended next steps:')
print('  - Tune Hybrid alpha/beta/gamma weights via grid search on a validation set.')
print('  - Add session-based context signals to improve cold-start performance.')
print('  - Consider intra-list diversity metrics (ILD) to reduce filter bubbles.')
print('=' * 65)